# 01 — PDF Extraction

1. Convert the 8 GNN PDFs in `data/raw/` into clean text chunks.
2. `data/processed/chunks.jsonl` — one JSON object per chunk.
3. `extract_text()` — raw text from PDF pages.
4. `clean_text()` — remove noise (hyphenation, page numbers, headers).
5. `chunk()` — sliding window chunking

In [34]:
import fitz
import json
import random
import sys

from collections import Counter
from dataclasses import asdict
from tqdm import tqdm

from pathlib import Path

In [18]:
sys.path.insert(0, '..')

In [19]:
from src.extraction.pdf_extractor import PDFExtractor

In [20]:
RAW_DIR = Path('../data/raw')
pdfs = sorted(RAW_DIR.glob('*.pdf'))

for p in pdfs:
    print(p.name)

1403.6652v2.pdf
1607.00653v1.pdf
1609.02907v4.pdf
1704.01212v2.pdf
1706.02216v4.pdf
1710.10903v3.pdf
1810.00826v3.pdf
1901.00596v4.pdf
the-illusion-of-thinking.pdf


## Step 1: Extract raw text from one PDF

In [21]:
extractor = PDFExtractor(chunk_size=512, chunk_overlap=64)

In [23]:
# test file path
SINGLE_FILE_PATH = Path('../data/raw/1403.6652v2.pdf')
result = extractor.process(SINGLE_FILE_PATH)

In [25]:
print(f"Extracted {len(result)} chunks from {SINGLE_FILE_PATH.name}")

Extracted 18 chunks from 1403.6652v2.pdf


In [26]:
result[0]

Chunk(text='DeepWalk: Online Learning of Social Representations Bryan Perozzi Stony Brook University Department of Computer Science Rami Al-Rfou Stony Brook University Department of Computer Science Steven Skiena Stony Brook University Department of Computer Science {bperozzi, ralrfou, skiena}@cs.stonybrook.edu ABSTRACT We present DeepWalk, a novel approach for learning latent representations of vertices in a network. These latent representations encode social relations in a continuous vector space, which is easily exploited by statistical models. DeepWalk generalizes recent advancements in language modeling and unsupervised feature learning (or deep learning) from sequences of words to graphs. DeepWalk uses local information obtained from truncated random walks to learn latent representations by treating walks as the equivalent of sentences. We demonstrate DeepWalk’s latent representations on several multi-label network classiﬁcation tasks for social networks such as BlogCatalog, Flic

## Step 2: Process all PDFs and save

In [28]:
# for all files

PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

In [29]:
all_chunks = []

for pdf in tqdm(pdfs):
    chunks = extractor.process(pdf)
    all_chunks.extend(chunks)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 10.09it/s]


In [30]:
with open(PROCESSED_DIR / 'chunks.jsonl', 'w') as f:
    for chunk in all_chunks:
        f.write(json.dumps(asdict(chunk)) + '\n')

print(f'Saved {len(all_chunks)} chunks.')

Saved 215 chunks.


## Step 3: quick check

In [32]:
chunks = [json.loads(l) for l in open(PROCESSED_DIR / 'chunks.jsonl')]

In [35]:
# chunks per paper
for paper, count in Counter(c['paper_id'] for c in chunks).items():
    print(f"{paper}: {count} chunks")

1403.6652v2: 18 chunks
1607.00653v1: 21 chunks
1609.02907v4: 16 chunks
1704.01212v2: 22 chunks
1706.02216v4: 25 chunks
1710.10903v3: 15 chunks
1810.00826v3: 24 chunks
1901.00596v4: 47 chunks
the-illusion-of-thinking: 27 chunks


In [39]:
# word counts
lengths = [len(c['text'].split()) for c in chunks]
print(f"min={min(lengths)} max={max(lengths)} avg={sum(lengths)//len(lengths)}")

min=84 max=512 avg=504


In [38]:
# sample
for c in random.sample(chunks, 3):
    print(f"\n{c['paper_id']} chunk {c['chunk_index']}")
    print(c['text'][:300])


1607.00653v1 chunk 3
from diverse domains, such as social networks, information networks, as well as networks from systems biology. Experiments demonstrate that node2vec outperforms state-of-the-art methods by up to 26.7% on multi-label classiﬁcation and up to 12.6% on link prediction. The algorithm shows competitive pe

the-illusion-of-thinking chunk 16
formatting and reasoning expectations. System Prompt - Tower of Hanoi You are a helpful assistant. Solve this puzzle for me. There are three pegs and n disks of different sizes stacked on the first peg. The disks are numbered from 1 (smallest) to n (largest). Disk moves in this puzzle should follow:

1901.00596v4 chunk 26
hidden representations. The majority of GAEs for graph generation are designed to solve the molecular graph generation problem, which has a high practical value in drug discovery. These methods either propose a new graph in a sequential manner or in a global manner. Sequential approaches generate a 


# Script Complete